In [1]:
"""
RET AlphaFold3 JSON Builder — alphafoldserver dialect

  (1) WT_ECD_dimer_GDNF_GFRa1 : RET ECD (1-700) dimer + GFRa1 + GDNF + glycans (NAG) + Ca2+
  (2) WT_KD_dimer_active      : RET kinase domain (721-1016) dimer, pY900/pY905 + ATP + Mg2+

All sequences are fetched from UniProt. All positions use UniProt numbering (1-based).
"""

import json
import os
import requests


# ══════════════════════════════════════════════════════════════════
# 0. Settings
# ══════════════════════════════════════════════════════════════════

UNIPROT = {
    "RET":   "P07949",
    "GFRA1": "P56159",
    "GDNF":  "P39905",
}

# ── ECD ─────────────────────────────────────────
ECD_START, ECD_END = 1, 700                       # UniProt 1-700
RET_NAG_POSITIONS  = [98, 151, 199, 336, 343, 361, 367, 377, 394, 448]
CA_PER_RET         = 4
ECD_COUNT          = 2                            # dimer

# ── Kinase ──────────────────────────────────────
KINASE_START, KINASE_END = 721, 1016              # UniProt 721-1016
KINASE_COUNT             = 2                      # active dimer
PTYR_POSITIONS           = [900, 905]             # activation-loop pTyr (UniProt)

# ── Output directory ────────────────────────────
JSON_DIR = "RET_AF3_json"


# ══════════════════════════════════════════════════════════════════
# 1. UniProt utilities
# ══════════════════════════════════════════════════════════════════

def fetch_uniprot_fasta(acc: str) -> str:
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta")
    r.raise_for_status()
    return "".join(r.text.strip().split("\n")[1:])


def get_mature_range(acc: str) -> tuple:
    """
    Return (start, end) of the UniProt 'Chain' feature (1-based, inclusive).

    Taking the max end of Signal/Propeptide fails for proteins with a
    C-terminal propeptide (e.g. the GPI-anchor propeptide of GFRa1),
    so the mature Chain range is used instead.
    """
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json")
    r.raise_for_status()
    chains = [f for f in r.json().get("features", []) if f.get("type") == "Chain"]
    assert chains, f"{acc}: no Chain feature found"
    best = max(chains, key=lambda f: f["location"]["end"]["value"]
                                     - f["location"]["start"]["value"])
    return best["location"]["start"]["value"], best["location"]["end"]["value"]


def slice_uniprot(seq: str, start: int, end: int) -> str:
    """Slice a UniProt 1-based, inclusive range."""
    return seq[start - 1:end]


# ══════════════════════════════════════════════════════════════════
# 2. alphafoldserver JSON utilities
# ══════════════════════════════════════════════════════════════════

def protein(seq: str, count: int, glycan_positions=None, ptms=None) -> dict:
    entry = {"sequence": seq, "count": count, "modifications": ptms or []}
    if glycan_positions:
        entry["glycans"] = [{"residues": "NAG", "position": p}
                            for p in glycan_positions]
    return {"proteinChain": entry}


def ion(code: str, count: int) -> dict:
    return {"ion": {"ion": code, "count": count}}


def ligand(ccd: str, count: int) -> dict:
    return {"ligand": {"ligand": ccd, "count": count}}


def write_job(name: str, sequences: list) -> str:
    job = [{"name": name, "modelSeeds": [], "sequences": sequences,
            "dialect": "alphafoldserver", "version": 1}]
    os.makedirs(JSON_DIR, exist_ok=True)
    path = os.path.join(JSON_DIR, f"{name}.json")
    with open(path, "w") as f:
        json.dump(job, f, indent=2)
    return path


# ══════════════════════════════════════════════════════════════════
# 3. Fetch sequences
# ══════════════════════════════════════════════════════════════════

print("Fetching sequences from UniProt...")

RET_seq = fetch_uniprot_fasta(UNIPROT["RET"])

GFRA1_full   = fetch_uniprot_fasta(UNIPROT["GFRA1"])
gfra1_range  = get_mature_range(UNIPROT["GFRA1"])
GFRA1_mature = slice_uniprot(GFRA1_full, *gfra1_range)

GDNF_full    = fetch_uniprot_fasta(UNIPROT["GDNF"])
gdnf_range   = get_mature_range(UNIPROT["GDNF"])
GDNF_mature  = slice_uniprot(GDNF_full, *gdnf_range)

print(f"  RET   {UNIPROT['RET']}: {len(RET_seq)} aa (full length)")
print(f"  GFRa1 {UNIPROT['GFRA1']}: {len(GFRA1_mature)} aa (mature {gfra1_range[0]}-{gfra1_range[1]})")
print(f"  GDNF  {UNIPROT['GDNF']}: {len(GDNF_mature)} aa (mature {gdnf_range[0]}-{gdnf_range[1]})")
assert GFRA1_mature and GDNF_mature, "Mature sequence is empty"


# ══════════════════════════════════════════════════════════════════
# 4. WT_ECD_dimer_GDNF_GFRa1
# ══════════════════════════════════════════════════════════════════

print("\n── ECD ─────────────────────────────────────────")

ECD_WT = slice_uniprot(RET_seq, ECD_START, ECD_END)

# Convert UniProt positions to positions within the ECD slice
glycan_pos = [p - ECD_START + 1 for p in RET_NAG_POSITIONS]
bad = [p for p, g in zip(RET_NAG_POSITIONS, glycan_pos) if ECD_WT[g - 1] != "N"]
assert not bad, f"Glycosylation positions that are not Asn: {bad}"

ecd_path = write_job("WT_ECD_dimer_GDNF_GFRa1", [
    protein(ECD_WT, ECD_COUNT, glycan_positions=glycan_pos),   # RET ECD + glycans
    protein(GFRA1_mature, ECD_COUNT),                          # GFRa1
    protein(GDNF_mature, ECD_COUNT),                           # GDNF
    ion("CA", CA_PER_RET * ECD_COUNT),                         # Ca2+
])

ecd_tokens = (len(ECD_WT) + len(GFRA1_mature) + len(GDNF_mature)) * ECD_COUNT
print(f"✓ {ecd_path}")
print(f"  RET ECD {len(ECD_WT)} aa x {ECD_COUNT}, {len(glycan_pos)} glycans/chain, "
      f"{CA_PER_RET * ECD_COUNT} Ca")
print(f"  Total {ecd_tokens} aa  {'⚠️ >5120' if ecd_tokens > 5120 else '✅'}")


# ══════════════════════════════════════════════════════════════════
# 5. WT_KD_dimer_active
# ══════════════════════════════════════════════════════════════════

print("\n── Kinase ──────────────────────────────────────")

KD_WT = slice_uniprot(RET_seq, KINASE_START, KINASE_END)

# Phospho-Tyr in the activation loop (positions within the kinase slice)
bad = [p for p in PTYR_POSITIONS if KD_WT[p - KINASE_START] != "Y"]
assert not bad, f"Phosphorylation positions that are not Tyr: {bad}"
kd_ptms = [{"ptmType": "CCD_PTR", "ptmPosition": p - KINASE_START + 1}
           for p in PTYR_POSITIONS]

kd_path = write_job("WT_KD_dimer_active", [
    protein(KD_WT, KINASE_COUNT, ptms=kd_ptms),                # RET KD + pTyr
    ligand("CCD_ATP", KINASE_COUNT),                           # ATP
    ion("MG", 2 * KINASE_COUNT),                               # Mg2+
])

print(f"✓ {kd_path}")
print(f"  RET KD {len(KD_WT)} aa x {KINASE_COUNT}, "
      f"pY{'/pY'.join(map(str, PTYR_POSITIONS))}, "
      f"{KINASE_COUNT} ATP + {2 * KINASE_COUNT} Mg")

print(f"\nDone: {sorted(os.listdir(JSON_DIR))}")

Fetching sequences from UniProt...
  RET   P07949: 1114 aa (full length)
  GFRa1 P56159: 405 aa (mature 25-429)
  GDNF  P39905: 134 aa (mature 78-211)

── ECD ─────────────────────────────────────────
✓ RET_AF3_json/WT_ECD_dimer_GDNF_GFRa1.json
  RET ECD 700 aa x 2, 10 glycans/chain, 8 Ca
  Total 2478 aa  ✅

── Kinase ──────────────────────────────────────
✓ RET_AF3_json/WT_KD_dimer_active.json
  RET KD 296 aa x 2, pY900/pY905, 2 ATP + 4 Mg

Done: ['WT_ECD_dimer_GDNF_GFRa1.json', 'WT_KD_dimer_active.json']
